# Flex32 PSD weighted `x` notebook with target-space toggle

This notebook trains the same `Flex32` model on PSD glossy-conditioned data with:

- fixed `x` parameterization
- weighted image-space loss focused on highlight regions
- an additional diffuse reconstruction loss
- a configurable target space via `TARGET_MODE = 'difference'` or `TARGET_MODE = 'diffuse'`

Use `difference` to predict `glossy - diffuse`, or `diffuse` to predict the diffuse image directly. The periodic eval always reports diffuse reconstruction metrics so both modes can be judged on the same output space.


In [ ]:
from __future__ import annotations

import math
import random
import re
import sys
from collections import OrderedDict
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / 'Run_Training.py').exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / 'Run_Training.py').exists() and (candidate / 'Backbone.py').exists():
            ROOT = candidate
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import Backbone
import ParamDiffuser as diff


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = ROOT / 'models' / '32'
CHECKPOINTS_DIR = ROOT / 'checkpoints'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'repo root: {ROOT}')
print(f'device: {DEVICE}')


In [ ]:
SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 10
NOISE_STEPS = 200
EPOCHS = 2000
LR = 1e-4
FINAL_LR = 1e-5
WARMUP_EPOCHS = 100
EMA_DECAY = 0.9999

TRAIN_TYPE = 'x'
TARGET_MODE = 'difference'  # switch between 'difference' and 'diffuse'
LOSS_KIND = 'l1'  # 'l1' or 'mse'
HIGHLIGHT_WEIGHT = 4.0
HIGHLIGHT_QUANTILE = 0.98
RECON_LOSS_WEIGHT = 0.5

EVAL_EVERY = 200
CHECKPOINT_EVERY = 500
VAL_BATCHES = 4
VAL_CASE_INDEX = 0
EVAL_SEED = 123

PSD_ROOT = Path('/Users/27171653/Desktop/PhD/Highlight-modelling/PSD_Dataset/PSD_Dataset')
TRAIN_GLOSSY_DIR = PSD_ROOT / 'PSD_Train' / 'PSD_Train_specular'
TRAIN_DIFFUSE_DIR = PSD_ROOT / 'PSD_Train' / 'PSD_Train_diffuse'
VAL_GLOSSY_DIR = PSD_ROOT / 'PSD_val' / 'PSD_val_specular'
VAL_DIFFUSE_DIR = PSD_ROOT / 'PSD_val' / 'PSD_val_diffuse'

if TRAIN_TYPE != 'x':
    raise ValueError(f'This notebook is fixed to x-parameterization, got {TRAIN_TYPE!r}')
if TARGET_MODE not in {'difference', 'diffuse'}:
    raise ValueError(f"TARGET_MODE must be 'difference' or 'diffuse', got {TARGET_MODE!r}")
if LOSS_KIND not in {'l1', 'mse'}:
    raise ValueError(f"LOSS_KIND must be 'l1' or 'mse', got {LOSS_KIND!r}")

SAVE_STEM = f'PSD_WeightedX_{TARGET_MODE}_Flex32'
TRAINER_SAVE_PATH = CHECKPOINTS_DIR / f'{SAVE_STEM}_checkpoint'
FINAL_MODEL_PATH = MODELS_DIR / f'{SAVE_STEM}.pth'

print(f'PSD root: {PSD_ROOT}')
print(f'target mode: {TARGET_MODE}')
print(f'train type: {TRAIN_TYPE}')
print(f'final model path: {FINAL_MODEL_PATH}')


In [ ]:
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
PIL_BILINEAR = getattr(Image, 'Resampling', Image).BILINEAR


def normalize_image_key(name: str) -> str:
    stem = Path(name).stem.lower()
    stem = stem.replace('specular', '').replace('glossy', '').replace('diffuse', '')
    return re.sub(r'[^a-z0-9]+', '', stem)


def pair_image_paths(glossy_dir: Path, diffuse_dir: Path):
    glossy_candidates = [path for path in glossy_dir.iterdir() if path.suffix.lower() in VALID_EXTS]
    diffuse_candidates = [path for path in diffuse_dir.iterdir() if path.suffix.lower() in VALID_EXTS]

    glossy_files = {path.name: path for path in glossy_candidates}
    diffuse_files = {path.name: path for path in diffuse_candidates}
    exact_names = sorted(set(glossy_files) & set(diffuse_files))
    if exact_names:
        return [(glossy_files[name], diffuse_files[name], name) for name in exact_names]

    glossy_by_key = {}
    for path in glossy_candidates:
        key = normalize_image_key(path.name)
        if key in glossy_by_key:
            raise RuntimeError(f'Duplicate glossy normalized key {key!r} in {glossy_dir}')
        glossy_by_key[key] = path

    diffuse_by_key = {}
    for path in diffuse_candidates:
        key = normalize_image_key(path.name)
        if key in diffuse_by_key:
            raise RuntimeError(f'Duplicate diffuse normalized key {key!r} in {diffuse_dir}')
        diffuse_by_key[key] = path

    common_keys = sorted(set(glossy_by_key) & set(diffuse_by_key))
    if not common_keys:
        raise RuntimeError(f'No paired PSD samples found in {glossy_dir} and {diffuse_dir}')

    return [(glossy_by_key[key], diffuse_by_key[key], glossy_by_key[key].name) for key in common_keys]


def load_rgb_tensor(path: Path, image_size: int) -> torch.Tensor:
    image = Image.open(path).convert('RGB')
    image = image.resize((image_size, image_size), resample=PIL_BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1).contiguous()


def build_weight_map(difference: torch.Tensor, highlight_weight: float, quantile: float) -> torch.Tensor:
    highlight_strength = difference.abs().mean(dim=0, keepdim=True)
    scale = torch.quantile(highlight_strength.flatten(), quantile).clamp_min(1e-6)
    focus = (highlight_strength / scale).clamp(0.0, 1.0)
    return 1.0 + highlight_weight * focus


class PairedPSDTargetDataset(Dataset):
    def __init__(self, glossy_dir: Path, diffuse_dir: Path, image_size: int, target_mode: str, highlight_weight: float, highlight_quantile: float):
        self.glossy_dir = Path(glossy_dir)
        self.diffuse_dir = Path(diffuse_dir)
        self.image_size = int(image_size)
        self.target_mode = target_mode
        self.highlight_weight = float(highlight_weight)
        self.highlight_quantile = float(highlight_quantile)

        if not self.glossy_dir.exists():
            raise FileNotFoundError(f'Missing glossy directory: {self.glossy_dir}')
        if not self.diffuse_dir.exists():
            raise FileNotFoundError(f'Missing diffuse directory: {self.diffuse_dir}')

        self.samples = pair_image_paths(self.glossy_dir, self.diffuse_dir)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        glossy_path, diffuse_path, name = self.samples[idx]
        glossy = load_rgb_tensor(glossy_path, self.image_size)
        diffuse = load_rgb_tensor(diffuse_path, self.image_size)
        difference = glossy - diffuse
        target = difference if self.target_mode == 'difference' else diffuse
        weight_map = build_weight_map(difference, self.highlight_weight, self.highlight_quantile)
        meta = {
            'name': name,
            'glossy_path': str(glossy_path),
            'diffuse_path': str(diffuse_path),
        }
        return glossy, target, diffuse, difference, weight_map, meta


train_dataset = PairedPSDTargetDataset(
    TRAIN_GLOSSY_DIR,
    TRAIN_DIFFUSE_DIR,
    image_size=IMAGE_SIZE,
    target_mode=TARGET_MODE,
    highlight_weight=HIGHLIGHT_WEIGHT,
    highlight_quantile=HIGHLIGHT_QUANTILE,
)
val_dataset = PairedPSDTargetDataset(
    VAL_GLOSSY_DIR,
    VAL_DIFFUSE_DIR,
    image_size=IMAGE_SIZE,
    target_mode=TARGET_MODE,
    highlight_weight=HIGHLIGHT_WEIGHT,
    highlight_quantile=HIGHLIGHT_QUANTILE,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

fixed_val_condition, fixed_val_target, fixed_val_diffuse, fixed_val_difference, fixed_val_weight_map, fixed_val_meta = val_dataset[VAL_CASE_INDEX]
fixed_val_condition = fixed_val_condition.unsqueeze(0).to(DEVICE)
fixed_val_target = fixed_val_target.unsqueeze(0).to(DEVICE)
fixed_val_diffuse = fixed_val_diffuse.unsqueeze(0).to(DEVICE)
fixed_val_difference = fixed_val_difference.unsqueeze(0).to(DEVICE)

fixed_eval_generator = torch.Generator()
fixed_eval_generator.manual_seed(EVAL_SEED)
fixed_eval_noise = torch.randn(fixed_val_target.shape, generator=fixed_eval_generator, dtype=fixed_val_target.dtype).to(DEVICE)

print(f'train dataset size: {len(train_dataset)}')
print(f'val dataset size: {len(val_dataset)}')
print(f'train batches per epoch: {len(train_loader)}')
print(f'fixed val case: {fixed_val_meta["name"]}')
print(f'fixed condition shape: {tuple(fixed_val_condition.shape)}')
print(f'fixed target shape: {tuple(fixed_val_target.shape)}')
print(f'fixed diffuse shape: {tuple(fixed_val_diffuse.shape)}')
print(f'fixed difference shape: {tuple(fixed_val_difference.shape)}')


In [ ]:
def get_cosine_lambda(initial_lr: float, final_lr: float, epochs: int, warmup_epoch: int):
    def cosine_lambda(epoch_idx: int) -> float:
        if epoch_idx < warmup_epoch:
            return epoch_idx / max(warmup_epoch, 1)
        cosine = (math.cos((epoch_idx - warmup_epoch) / max(epochs - warmup_epoch, 1) * math.pi) + 1.0) / 2.0
        return 1.0 - (1.0 - cosine) * (1.0 - final_lr / initial_lr)

    return cosine_lambda


def checkpoint_save(model, optimizer, loss: float, epoch: int, save_path: Path, target_mode: str):
    model_dir = Path(f'{save_path}_epoch_{epoch}_loss_{loss:.4f}')
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'epoch': int(epoch),
            'loss': float(loss),
            'train_type': TRAIN_TYPE,
            'target_mode': target_mode,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        },
        model_dir / 'model.pth',
    )


def update_ema(ema_model, model, decay: float = 0.9999):
    ema_params = OrderedDict(ema_model.named_parameters())
    model_params = OrderedDict(model.named_parameters())
    for name, param in model_params.items():
        if name in ema_params:
            ema_params[name].data.mul_(decay).add_(param.data, alpha=1.0 - decay)


def weighted_pixel_loss(prediction: torch.Tensor, target: torch.Tensor, weight_map: torch.Tensor, loss_kind: str) -> torch.Tensor:
    if loss_kind == 'l1':
        return (weight_map * (prediction - target).abs()).mean()
    return (weight_map * (prediction - target).pow(2)).mean()


def prediction_to_diffuse(prediction: torch.Tensor, condition: torch.Tensor, target_mode: str) -> torch.Tensor:
    if target_mode == 'difference':
        return condition - prediction
    return prediction


model = Backbone.Flex(size=IMAGE_SIZE, noise_steps=NOISE_STEPS).to(DEVICE)
ema_model = deepcopy(model).to(DEVICE)
ema_model.load_state_dict(model.state_dict())
ema_model.eval()

diffuser = diff.CosSchDiffuser(steps=NOISE_STEPS, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=get_cosine_lambda(initial_lr=LR, final_lr=FINAL_LR, epochs=EPOCHS, warmup_epoch=WARMUP_EPOCHS),
)

num_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f'model: {model.__class__.__name__}')
print(f'trainable parameters: {num_params:,}')
print(f'optimizer: AdamW(lr={LR})')
print(f'diffuser: {diffuser.name}, steps={diffuser.steps}')


def compute_step_outputs(model: torch.nn.Module, batch, diffuser: diff.Diffuser, device: torch.device, target_mode: str, loss_kind: str, recon_weight: float):
    condition, targets, diffuse, difference, weight_map, _meta = batch
    condition = condition.to(device)
    targets = targets.to(device)
    diffuse = diffuse.to(device)
    weight_map = weight_map.to(device)

    batch_size = condition.shape[0]
    t = torch.randint(0, diffuser.steps, (batch_size,), dtype=torch.long, device=device)
    noise = torch.randn_like(targets)
    noisy_xt = diffuser.forward_diffusion(targets, t, noise)
    prediction = model(noisy_xt, t, condition)

    target_loss = weighted_pixel_loss(prediction, targets, weight_map, loss_kind)
    predicted_diffuse = prediction_to_diffuse(prediction, condition, target_mode)
    recon_loss = F.l1_loss(predicted_diffuse, diffuse)
    total_loss = target_loss + recon_weight * recon_loss

    return {
        'total_loss': total_loss,
        'target_loss': target_loss.detach(),
        'recon_loss': recon_loss.detach(),
        'prediction': prediction,
        'target': targets,
        'diffuse': diffuse,
        'condition': condition,
    }


def train_step(model: torch.nn.Module, batch, diffuser: diff.Diffuser, device: torch.device, target_mode: str, loss_kind: str, recon_weight: float) -> torch.Tensor:
    return compute_step_outputs(model, batch, diffuser, device, target_mode, loss_kind, recon_weight)['total_loss']


def evaluate_objective(model: torch.nn.Module, loader, diffuser: diff.Diffuser, device: torch.device, target_mode: str, loss_kind: str, recon_weight: float, max_batches: int | None = None) -> dict:
    was_training = model.training
    model.eval()
    totals, targets, recons = [], [], []
    with torch.no_grad():
        for idx, batch in enumerate(loader):
            if max_batches is not None and idx >= max_batches:
                break
            outputs = compute_step_outputs(model, batch, diffuser, device, target_mode, loss_kind, recon_weight)
            totals.append(float(outputs['total_loss'].item()))
            targets.append(float(outputs['target_loss'].item()))
            recons.append(float(outputs['recon_loss'].item()))
    if was_training:
        model.train()
    return {
        'objective': float(np.mean(totals)) if totals else float('nan'),
        'target_loss': float(np.mean(targets)) if targets else float('nan'),
        'recon_loss': float(np.mean(recons)) if recons else float('nan'),
    }


def channel_limits(a: torch.Tensor, b: torch.Tensor):
    low = min(float(a.min()), float(b.min()))
    high = max(float(a.max()), float(b.max()))
    if abs(high - low) < 1e-8:
        high = low + 1e-8
    return low, high


def normalize_rgb_pair(target_rgb: torch.Tensor, prediction_rgb: torch.Tensor):
    pair = torch.stack([target_rgb, prediction_rgb], dim=0)
    low = pair.amin(dim=(0, 2, 3), keepdim=True)
    high = pair.amax(dim=(0, 2, 3), keepdim=True)
    scale = (high - low).clamp_min(1e-6)
    target_norm = ((target_rgb.unsqueeze(0) - low) / scale).squeeze(0).clamp(0.0, 1.0)
    prediction_norm = ((prediction_rgb.unsqueeze(0) - low) / scale).squeeze(0).clamp(0.0, 1.0)
    return target_norm, prediction_norm


def plot_prediction(condition_tensor: torch.Tensor, target_tensor: torch.Tensor, prediction_tensor: torch.Tensor, true_diffuse_tensor: torch.Tensor, target_mode: str, epoch: int):
    condition = condition_tensor.detach().cpu().squeeze(0)
    target = target_tensor.detach().cpu().squeeze(0)
    prediction = prediction_tensor.detach().cpu().squeeze(0)
    true_diffuse = true_diffuse_tensor.detach().cpu().squeeze(0)
    predicted_diffuse = prediction_to_diffuse(prediction, condition, target_mode).clamp(0.0, 1.0)

    target_rgb, prediction_rgb = normalize_rgb_pair(target, prediction)
    target_error = (prediction - target).abs()
    diffuse_error = (predicted_diffuse - true_diffuse).abs().mean(dim=0)
    target_label = 'Difference' if target_mode == 'difference' else 'Diffuse'

    fig, axes = plt.subplots(5, 3, figsize=(12, 18))
    for channel_idx, channel_name in enumerate(['Red', 'Green', 'Blue']):
        vmin, vmax = channel_limits(target[channel_idx], prediction[channel_idx])
        axes[channel_idx, 0].imshow(target[channel_idx], cmap='coolwarm', vmin=vmin, vmax=vmax)
        axes[channel_idx, 0].set_title(f'{channel_name} {target_label} target')
        axes[channel_idx, 1].imshow(prediction[channel_idx], cmap='coolwarm', vmin=vmin, vmax=vmax)
        axes[channel_idx, 1].set_title(f'{channel_name} {target_label} prediction')
        axes[channel_idx, 2].imshow(target_error[channel_idx], cmap='magma')
        axes[channel_idx, 2].set_title(f'{channel_name} abs error')

    axes[3, 0].imshow(target_rgb.permute(1, 2, 0).numpy())
    axes[3, 0].set_title(f'RGB {target_label.lower()} target')
    axes[3, 1].imshow(prediction_rgb.permute(1, 2, 0).numpy())
    axes[3, 1].set_title(f'RGB {target_label.lower()} prediction')
    axes[3, 2].imshow(target_error.mean(dim=0).numpy(), cmap='magma')
    axes[3, 2].set_title('RGB mean abs error')

    axes[4, 0].imshow(condition.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[4, 0].set_title('Condition glossy')
    axes[4, 1].imshow(true_diffuse.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[4, 1].set_title('True diffuse')
    axes[4, 2].imshow(predicted_diffuse.permute(1, 2, 0).numpy())
    axes[4, 2].set_title('Predicted diffuse')

    for ax in axes.ravel():
        ax.axis('off')

    fig.suptitle(f'Fixed validation inference at epoch {epoch} | x-param | {target_mode} target', fontsize=16)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(diffuse_error.numpy(), cmap='magma')
    ax.set_title('Diffuse abs error')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def run_eval(model: torch.nn.Module, loader, diffuser: diff.Diffuser, fixed_condition: torch.Tensor, fixed_target: torch.Tensor, fixed_diffuse: torch.Tensor, fixed_noise: torch.Tensor, device: torch.device, epoch: int, target_mode: str, loss_kind: str, recon_weight: float, max_batches: int | None = None):
    objective_metrics = evaluate_objective(model, loader, diffuser, device, target_mode, loss_kind, recon_weight, max_batches=max_batches)
    with torch.no_grad():
        prediction = diffuser.sample_from_noise(
            model,
            fixed_condition,
            parameterization='x',
            show_progress=False,
            initial_noise=fixed_noise,
        )

    target = fixed_target.squeeze(0)
    condition = fixed_condition.squeeze(0)
    true_diffuse = fixed_diffuse.squeeze(0)
    predicted_diffuse = prediction_to_diffuse(prediction.squeeze(0), condition, target_mode)

    target_mse = float(F.mse_loss(prediction.squeeze(0), target).item())
    target_mae = float((prediction.squeeze(0) - target).abs().mean().item())
    diffuse_mse = float(F.mse_loss(predicted_diffuse, true_diffuse).item())
    diffuse_mae = float((predicted_diffuse - true_diffuse).abs().mean().item())
    diffuse_psnr = float((10.0 * torch.log10(1.0 / F.mse_loss(predicted_diffuse.clamp(0.0, 1.0), true_diffuse).clamp_min(1e-10))).item())

    plot_prediction(fixed_condition, fixed_target, prediction, fixed_diffuse, target_mode=target_mode, epoch=epoch)
    return {
        'epoch': epoch,
        'val_objective': objective_metrics['objective'],
        'val_target_loss': objective_metrics['target_loss'],
        'val_recon_loss': objective_metrics['recon_loss'],
        'target_mse': target_mse,
        'target_mae': target_mae,
        'diffuse_mse': diffuse_mse,
        'diffuse_mae': diffuse_mae,
        'diffuse_psnr': diffuse_psnr,
    }


In [ ]:
progress_bar = tqdm(total=EPOCHS * len(train_loader), desc=f'Training [x | {TARGET_MODE}]', dynamic_ncols=True)
train_loss_history = []
train_target_loss_history = []
train_recon_loss_history = []
eval_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_total = 0.0
    epoch_target = 0.0
    epoch_recon = 0.0

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        outputs = compute_step_outputs(model, batch, diffuser, DEVICE, TARGET_MODE, LOSS_KIND, RECON_LOSS_WEIGHT)
        loss = outputs['total_loss']
        loss.backward()
        optimizer.step()
        update_ema(ema_model, model, decay=EMA_DECAY)

        epoch_total += float(outputs['total_loss'].item())
        epoch_target += float(outputs['target_loss'].item())
        epoch_recon += float(outputs['recon_loss'].item())
        progress_bar.update(1)
        progress_bar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')

    epoch_total /= len(train_loader)
    epoch_target /= len(train_loader)
    epoch_recon /= len(train_loader)
    train_loss_history.append(epoch_total)
    train_target_loss_history.append(epoch_target)
    train_recon_loss_history.append(epoch_recon)
    scheduler.step()

    print(f'Epoch {epoch:4d} | total {epoch_total:.6f} | target {epoch_target:.6f} | recon {epoch_recon:.6f}')

    if epoch % CHECKPOINT_EVERY == 0:
        checkpoint_save(model, optimizer, epoch_total, epoch, TRAINER_SAVE_PATH, TARGET_MODE)

    if epoch % EVAL_EVERY == 0:
        metrics = run_eval(
            ema_model,
            val_loader,
            diffuser,
            fixed_val_condition,
            fixed_val_target,
            fixed_val_diffuse,
            fixed_eval_noise,
            DEVICE,
            epoch,
            target_mode=TARGET_MODE,
            loss_kind=LOSS_KIND,
            recon_weight=RECON_LOSS_WEIGHT,
            max_batches=VAL_BATCHES,
        )
        eval_history.append(metrics)
        print(
            f"Eval {epoch:4d} | val objective {metrics['val_objective']:.6f} | "
            f"target mse {metrics['target_mse']:.6f} | diffuse mse {metrics['diffuse_mse']:.6f} | "
            f"diffuse psnr {metrics['diffuse_psnr']:.4f} dB"
        )

progress_bar.close()

torch.save(
    {
        'epoch': EPOCHS,
        'train_type': TRAIN_TYPE,
        'target_mode': TARGET_MODE,
        'model_state_dict': model.state_dict(),
        'ema_model_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss_history': train_loss_history,
        'train_target_loss_history': train_target_loss_history,
        'train_recon_loss_history': train_recon_loss_history,
        'eval_history': eval_history,
    },
    FINAL_MODEL_PATH,
)
print(f'Final model saved to {FINAL_MODEL_PATH}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, len(train_loss_history) + 1), train_loss_history, label='total loss')
axes[0].plot(range(1, len(train_target_loss_history) + 1), train_target_loss_history, label='weighted target loss')
axes[0].plot(range(1, len(train_recon_loss_history) + 1), train_recon_loss_history, label='recon loss')
axes[0].set_title(f'Train Losses ({TARGET_MODE})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)
axes[0].legend()

if eval_history:
    eval_epochs = [item['epoch'] for item in eval_history]
    axes[1].plot(eval_epochs, [item['val_objective'] for item in eval_history], label='val objective')
    axes[1].plot(eval_epochs, [item['diffuse_mse'] for item in eval_history], label='diffuse mse')
    axes[1].plot(eval_epochs, [item['diffuse_psnr'] for item in eval_history], label='diffuse psnr')
    axes[1].set_title(f'Validation Curves ({TARGET_MODE})')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Metric')
    axes[1].grid(True)
    axes[1].legend()
else:
    axes[1].axis('off')

plt.tight_layout()
plt.show()
